In [2]:
import os
import json
import requests

from dotenv import load_dotenv
from jsonschema import validate, ValidationError

load_dotenv()

api_key = os.getenv("LLM_API_KEY")

print("API Loaded:", api_key is not None)

API Loaded: True


In [3]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [4]:
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

response = call_llm(system_prompt, user_prompt)

print(response)

hello


In [6]:
import os

print(os.getcwd())

c:\Users\Rethi\OneDrive\Desktop\Neurosage\notebooks


In [7]:
import pandas as pd

df = pd.read_csv("../outputs/cleaned_data.csv")

print(df.shape)
df.head()

(2149, 35)


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


In [8]:
# Select three patient records

records = df.iloc[[0, 1, 2]]

records

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid


In [9]:
import json

record_list = records.to_dict(orient="records")

print(json.dumps(record_list[0], indent=4))

{
    "PatientID": 4751,
    "Age": 73,
    "Gender": 0,
    "Ethnicity": 0,
    "EducationLevel": 2,
    "BMI": 22.927749230993864,
    "Smoking": 0,
    "AlcoholConsumption": 13.29721772827684,
    "PhysicalActivity": 6.327112473553353,
    "DietQuality": 1.3472143059081076,
    "SleepQuality": 9.025678665766115,
    "FamilyHistoryAlzheimers": 0,
    "CardiovascularDisease": 0,
    "Diabetes": 1,
    "Depression": 1,
    "HeadInjury": 0,
    "Hypertension": 0,
    "SystolicBP": 142,
    "DiastolicBP": 72,
    "CholesterolTotal": 242.3668396963656,
    "CholesterolLDL": 56.15089696091113,
    "CholesterolHDL": 33.68256349839592,
    "CholesterolTriglycerides": 162.18914307736603,
    "MMSE": 21.46353236431666,
    "FunctionalAssessment": 6.518876973217633,
    "MemoryComplaints": 0,
    "BehavioralProblems": 0,
    "ADL": 1.7258834599441897,
    "Confusion": 0,
    "Disorientation": 0,
    "PersonalityChanges": 0,
    "DifficultyCompletingTasks": 1,
    "Forgetfulness": 0,
    "Diagno

In [10]:
system_prompt = """
You are an Alzheimer's patient risk assessment assistant.

Your task is to assess one patient record and return ONLY valid JSON.

Use the following rubric:

- risk_tier: low, medium, or high
- flag_for_review: true or false
- primary_signal: the strongest reason for the assessment
- confidence: low, medium, or high
- recommended_action: one short recommendation

Return ONLY valid JSON.

Worked Example

Input:
{
 "Age":82,
 "MMSE":15,
 "MemoryComplaints":1,
 "Confusion":1,
 "Diagnosis":1
}

Output:
{
 "risk_tier":"high",
 "flag_for_review":true,
 "primary_signal":"Low MMSE with confirmed diagnosis",
 "confidence":"high",
 "recommended_action":"Refer to neurologist for comprehensive evaluation"
}
"""

In [11]:
user_prompt = f"""
Assess the following patient record.

Return ONLY valid JSON.

Patient Record:

{json.dumps(record_list[0], indent=2)}
"""

In [12]:
response = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)

```json
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and high cholesterol with some cognitive difficulties",
  "confidence": "medium",
  "recommended_action": "Monitor cognitive decline and manage diabetes"
}
```


In [13]:
schema = {
    "type": "object",
    "properties": {
        "risk_tier": {
            "type": "string"
        },
        "flag_for_review": {
            "type": "boolean"
        },
        "primary_signal": {
            "type": "string"
        },
        "confidence": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "risk_tier",
        "flag_for_review",
        "primary_signal",
        "confidence",
        "recommended_action"
    ]
}

In [14]:
response = response.strip()

print(response)

```json
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and high cholesterol with some cognitive difficulties",
  "confidence": "medium",
  "recommended_action": "Monitor cognitive decline and manage diabetes"
}
```


In [15]:
response = response.replace("```json", "")
response = response.replace("```", "")
response = response.strip()

In [16]:
assessment = json.loads(response)

assessment

{'risk_tier': 'medium',
 'flag_for_review': True,
 'primary_signal': 'Diabetes and high cholesterol with some cognitive difficulties',
 'confidence': 'medium',
 'recommended_action': 'Monitor cognitive decline and manage diabetes'}

In [17]:
try:
    validate(instance=assessment, schema=schema)
    print("✅ Validation Passed")
except ValidationError as e:
    print("❌ Validation Failed")
    print(e)

✅ Validation Passed


In [18]:
fallback = {
    "risk_tier": None,
    "flag_for_review": None,
    "primary_signal": None,
    "confidence": None,
    "recommended_action": None
}

try:
    assessment = json.loads(response)
    validate(instance=assessment, schema=schema)
    print("✅ Validation Passed")
    print(assessment)

except (json.JSONDecodeError, ValidationError) as e:
    print("❌ Validation Failed")
    print(e)
    assessment = fallback

print("\nFinal Output:")
print(assessment)

✅ Validation Passed
{'risk_tier': 'medium', 'flag_for_review': True, 'primary_signal': 'Diabetes and high cholesterol with some cognitive difficulties', 'confidence': 'medium', 'recommended_action': 'Monitor cognitive decline and manage diabetes'}

Final Output:
{'risk_tier': 'medium', 'flag_for_review': True, 'primary_signal': 'Diabetes and high cholesterol with some cognitive difficulties', 'confidence': 'medium', 'recommended_action': 'Monitor cognitive decline and manage diabetes'}


In [19]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [20]:
test1 = "Email: rethi@gmail.com"

test2 = "Age:73 BMI:22 MMSE:21"

print("Test 1:", has_pii(test1))
print("Test 2:", has_pii(test2))

Test 1: True
Test 2: False


In [21]:
user_input = "Email: rethi@gmail.com"

if has_pii(user_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, user_input))

Input blocked: PII detected.


In [22]:
user_input = json.dumps(record_list[1], indent=2)

if has_pii(user_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, user_input))

```
{
  "risk_tier": "low",
  "flag_for_review": false,
  "primary_signal": "Normal MMSE and no significant memory complaints",
  "confidence": "medium",
  "recommended_action": "Monitor patient's condition and reassess in 6 months"
}
```


In [23]:
for i, record in enumerate(record_list):

    user_prompt = f"""
Assess the following patient.

Return ONLY valid JSON.

{json.dumps(record, indent=2)}
"""

    response = call_llm(system_prompt, user_prompt)

    print("=" * 60)
    print(f"Patient {i+1}")
    print(response)

Patient 1
```
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and high cholesterol with some cognitive difficulties",
  "confidence": "medium",
  "recommended_action": "Monitor cognitive decline and manage diabetes"
}
```
Patient 2
```json
{
  "risk_tier": "low",
  "flag_for_review": false,
  "primary_signal": "Normal MMSE and no significant memory complaints",
  "confidence": "medium",
  "recommended_action": "Monitor cognitive function and lifestyle factors"
}
```
Patient 3
```
{
  "risk_tier": "high",
  "flag_for_review": true,
  "primary_signal": "Low MMSE score with disorientation and difficulty completing tasks",
  "confidence": "high",
  "recommended_action": "Refer to neurologist for comprehensive evaluation"
}
```


In [24]:
patient = json.dumps(record_list[0], indent=2)

response0 = call_llm(system_prompt, patient, temperature=0)

response07 = call_llm(system_prompt, patient, temperature=0.7)

print("Temperature = 0")
print(response0)

print("\n")

print("Temperature = 0.7")
print(response07)

Temperature = 0
```
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and DifficultyCompletingTasks",
  "confidence": "medium",
  "recommended_action": "Monitor cognitive decline and manage diabetes"
}
```


Temperature = 0.7
```
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and high total cholesterol with some difficulty completing tasks",
  "confidence": "medium",
  "recommended_action": "Monitor and manage diabetes and cholesterol levels"
}
```


In [25]:
patient = json.dumps(record_list[0], indent=2)

response0 = call_llm(system_prompt, patient, temperature=0)

response07 = call_llm(system_prompt, patient, temperature=0.7)

print("Temperature = 0")
print(response0)

print("\n")

print("Temperature = 0.7")
print(response07)

Temperature = 0
```
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and Depression with moderate MMSE score",
  "confidence": "medium",
  "recommended_action": "Schedule follow-up appointment for further evaluation"
}
```


Temperature = 0.7
```json
{
  "risk_tier": "medium",
  "flag_for_review": true,
  "primary_signal": "Diabetes and high cholesterol levels with reported difficulty completing tasks",
  "confidence": "medium",
  "recommended_action": "Schedule a follow-up appointment for further evaluation"
}
```


In [28]:
results = []

for i, record in enumerate(record_list):

    user_prompt = f"""
Assess the following patient record.

Return ONLY valid JSON.

{json.dumps(record, indent=2)}
"""

    if has_pii(user_prompt):
        results.append({
            "Patient": i + 1,
            "LLM_Output": "Blocked",
            "Validation": "Not Run",
            "Guardrail": "Blocked"
        })
        continue

    response = call_llm(system_prompt, user_prompt, temperature=0)

    clean_response = response.replace("```json", "").replace("```", "").strip()

    try:
        parsed = json.loads(clean_response)
        validate(instance=parsed, schema=schema)

        validation = "Pass"

    except Exception:
        parsed = fallback
        validation = "Fail"

    results.append({
        "Patient": i + 1,
        "LLM_Output": parsed,
        "Validation": validation,
        "Guardrail": "Passed"
    })

print(results)

[{'Patient': 1, 'LLM_Output': {'risk_tier': 'medium', 'flag_for_review': True, 'primary_signal': 'Diabetes and high cholesterol with some cognitive difficulties', 'confidence': 'medium', 'recommended_action': 'Monitor cognitive function and manage diabetes and cholesterol'}, 'Validation': 'Pass', 'Guardrail': 'Passed'}, {'Patient': 2, 'LLM_Output': {'risk_tier': 'low', 'flag_for_review': False, 'primary_signal': 'Normal MMSE and no significant memory complaints', 'confidence': 'medium', 'recommended_action': 'Monitor cognitive function and lifestyle factors'}, 'Validation': 'Pass', 'Guardrail': 'Passed'}, {'Patient': 3, 'LLM_Output': {'risk_tier': 'high', 'flag_for_review': True, 'primary_signal': 'Low MMSE score with disorientation and difficulty completing tasks', 'confidence': 'high', 'recommended_action': 'Refer to neurologist for comprehensive evaluation'}, 'Validation': 'Pass', 'Guardrail': 'Passed'}]


In [29]:
results_df = pd.DataFrame(results)

results_df

,Patient,LLM_Output,Validation,Guardrail
0,1,"{'risk_tier': 'medium', 'flag_for_review': Tru...",Pass,Passed
1,2,"{'risk_tier': 'low', 'flag_for_review': False,...",Pass,Passed
2,3,"{'risk_tier': 'high', 'flag_for_review': True,...",Pass,Passed
